<a href="https://colab.research.google.com/github/pranay778/A.Pranay-Durgesh-Varma-192425094/blob/main/NLP_IMPLEMENTATION_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import re
import math
import numpy as np

# Install PySide6 if not already installed
try:
    from PySide6.QtWidgets import (
        QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
        QTextEdit, QPushButton, QLabel, QFrame, QMessageBox,
        QProgressBar, QStackedWidget
    )
    from PySide6.QtCore import Qt
    from PySide6.QtGui import QFont
except ImportError:
    print("PySide6 not found. Installing...")
    !pip install PySide6
    from PySide6.QtWidgets import (
        QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
        QTextEdit, QPushButton, QLabel, QFrame, QMessageBox,
        QProgressBar, QStackedWidget
    )
    from PySide6.QtCore import Qt
    from PySide6.QtGui import QFont

from sklearn.ensemble import RandomForestRegressor


# ============================================================
# MACHINE LEARNING MODEL
# ============================================================

def create_ml_model():

    # Training data:
    # [words, sentences, avg_sentence_length,
    #  grammar_errors, vocabulary, keyword_score, coherence]

    X = np.array([
        [80, 5, 16, 8, 40, 45, 45],
        [100, 6, 16, 6, 50, 55, 50],
        [120, 7, 17, 5, 55, 60, 55],
        [150, 8, 19, 4, 60, 70, 65],
        [180, 9, 20, 3, 65, 75, 70],
        [220, 10, 22, 2, 70, 80, 75],
        [250, 11, 23, 2, 75, 85, 80],
        [300, 12, 25, 1, 80, 90, 85],
        [350, 14, 25, 1, 85, 95, 90],
        [400, 16, 25, 0, 90, 98, 95],
        # Additional data points
        [50, 4, 12, 10, 30, 35, 30],
        [70, 5, 14, 7, 45, 50, 40],
        [90, 6, 15, 5, 50, 58, 52],
        [110, 7, 18, 3, 58, 65, 60],
        [130, 8, 19, 2, 62, 70, 68],
        [160, 9, 21, 1, 68, 78, 72],
        [200, 10, 23, 1, 72, 82, 78],
        [280, 11, 24, 0, 78, 88, 82],
        [320, 13, 25, 0, 82, 92, 88],
        [380, 15, 26, 0, 88, 96, 92]
    ])

    y = np.array([
        45, 50, 55, 60, 65,
        70, 75, 80, 90, 95,
        # Additional data points
        35, 48, 53, 62, 69,
        73, 79, 84, 89, 93
    ])

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    model.fit(X, y)

    return model


model = create_ml_model()


# ============================================================
# NLP FUNCTIONS
# ============================================================

def get_sentences(text):

    sentences = re.split(r'[.!?]+', text)

    return [
        s.strip()
        for s in sentences
        if s.strip()
    ]


def get_words(text):

    return re.findall(r'\b[a-zA-Z]+\b', text.lower())


def grammar_analysis(text):

    words = get_words(text)

    errors = []

    # Simple grammar patterns
    patterns = {
        r'\bi am agree\b': "I agree",
        r'\bhe go\b': "he goes",
        r'\bshe go\b': "she goes",
        r'\bthey goes\b': "they go",
        r'\bhe have\b': "he has",
        r'\bshe have\b': "she has",
        r'\bvery very\b': "very",
        r'\bdid not went\b': "did not go",
        r'\bmore better\b': "better",
        r'\bdiscuss about\b': "discuss"
    }

    for pattern, correction in patterns.items():

        matches = re.findall(pattern, text.lower())

        for match in matches:
            errors.append(
                f"'{match}' \u2192 Suggested: '{correction}'"
            )

    # Repeated words
    for i in range(len(words) - 1):

        if words[i] == words[i + 1]:

            errors.append(
                f"Repeated word: '{words[i]}'"
            )

    # Score
    word_count = max(len(words), 1)

    penalty = len(errors) * 5

    score = max(
        0,
        min(100, 100 - penalty)
    )

    return score, errors


def vocabulary_score(text):

    words = get_words(text)

    if not words:
        return 0

    unique_words = set(words)

    richness = len(unique_words) / len(words)

    score = min(100, richness * 140)

    return round(score)


def coherence_score(text):

    sentences = get_sentences(text)

    if len(sentences) < 2:
        return 40

    lengths = [
        len(get_words(sentence))
        for sentence in sentences
    ]

    average = sum(lengths) / len(lengths)

    # Reasonable sentence length
    if 10 <= average <= 25:
        score = 90

    elif 7 <= average <= 30:
        score = 75

    else:
        score = 60

    # Check transition words
    transitions = [
        "however",
        "therefore",
        "moreover",
        "furthermore",
        "because",
        "although",
        "firstly",
        "finally",
        "also",
        "thus",
        "in addition"
    ]

    text_lower = text.lower()

    count = sum(
        1 for word in transitions
        if word in text_lower
    )

    score += min(count * 2, 10)

    return min(score, 100)


def content_score(text):

    words = get_words(text)

    if len(words) < 20:
        return 30

    # Basic content depth estimation
    paragraphs = text.split("\n")

    paragraph_score = min(
        len(paragraphs) * 15,
        40
    )

    word_score = min(
        len(words) / 3,
        40
    )

    keyword_score = 20

    score = (
        paragraph_score +
        word_score +
        keyword_score
    )

    return min(round(score), 100)


def get_grade(score):

    if score >= 90:
        return "A+"

    elif score >= 80:
        return "A"

    elif score >= 70:
        return "B"

    elif score >= 60:
        return "C"

    elif score >= 50:
        return "D"

    else:
        return "F"


def generate_feedback(
        grammar,
        content,
        coherence,
        vocabulary,
        errors
):

    feedback = []

    # Strengths
    feedback.append("\u2713 STRENGTHS")

    if grammar >= 80:
        feedback.append(
            "\u2022 Good grammatical accuracy."
        )
    else:
        feedback.append(
            "\u2022 Grammar needs improvement."
        )

    if vocabulary >= 75:
        feedback.append(
            "\u2022 Good vocabulary usage."
        )
    else:
        feedback.append(
            "\u2022 Try using a wider range of vocabulary."
        )

    if content >= 75:
        feedback.append(
            "\u2022 Essay contains relevant content."
        )
    else:
        feedback.append(
            "\u2022 Add more supporting ideas and examples."
        )

    if coherence >= 75:
        feedback.append(
            "\u2022 Ideas are reasonably well connected."
        )
    else:
        feedback.append(
            "\u2022 Improve connections between paragraphs."
        )

    feedback.append("")
    feedback.append("\u26a0 AREAS TO IMPROVE")

    if errors:

        for error in errors[:5]:

            feedback.append(
                "\u2022 " + error
            )

    else:

        feedback.append(
            "\u2022 No major grammar problems detected."
        )

    if coherence < 75:

        feedback.append(
            "\u2022 Use transition words such as however, therefore and moreover."
        )

    if vocabulary < 75:

        feedback.append(
            "\u2022 Avoid repeating the same words."
        )

    if content < 75:

        feedback.append(
            "\u2022 Add examples, arguments and explanations."
        )

    feedback.append("")
    feedback.append("\ud83d\udca1 RECOMMENDATIONS")

    feedback.append(
        "1. Review grammatical mistakes."
    )

    feedback.append(
        "2. Add stronger supporting examples."
    )

    feedback.append(
        "3. Improve paragraph transitions."
    )

    feedback.append(
        "4. Use more academic vocabulary."
    )

    feedback.append(
        "5. Write a clear conclusion."
    )

    return "\n".join(feedback)


# ============================================================
# MAIN GUI
# ============================================================

class EssayAssessment(QMainWindow):

    def __init__(self):

        super().__init__()

        self.setWindowTitle(
            "Intelligent Essay Assessment - NLP & ML"
        )

        self.setMinimumSize(1200, 750)

        self.setStyleSheet("""
            QMainWindow {
                background-color: #0f172a;
            }

            QLabel {
                color: white;
            }

            QTextEdit {
                background-color: #1e293b;
                color: white;
                border: 1px solid #334155;
                border-radius: 10px;
                padding: 15px;
                font-size: 15px;
            }

            QPushButton {
                background-color: #2563eb;
                color: white;
                border: none;
                border-radius: 8px;
                padding: 12px 20px;
                font-size: 14px;
                font-weight: bold;
            }

            QPushButton:hover {
                background-color: #3b82f6;
            }

            QProgressBar {
                border: none;
                border-radius: 7px;
                background-color: #334155;
                height: 12px;
            }

            QProgressBar::chunk {
                background-color: #22c55e;
                border-radius: 7px;
            }
        """)

        self.setup_ui()


    # ========================================================
    # UI
    # ========================================================

    def setup_ui(self):

        main_widget = QWidget()

        main_layout = QHBoxLayout(
            main_widget
        )

        # ----------------------------------------------------
        # SIDEBAR
        # ----------------------------------------------------

        sidebar = QFrame()

        sidebar.setFixedWidth(220)

        sidebar.setStyleSheet("""
            QFrame {
                background-color: #111827;
                border-radius: 10px;
            }
        """)

        side_layout = QVBoxLayout(sidebar)

        title = QLabel("\ud83e\udde0 EssayAI")

        title.setFont(
            QFont("Arial", 22, QFont.Bold)
        )

        side_layout.addWidget(title)

        subtitle = QLabel(
            "NLP + Machine Learning"
        )

        subtitle.setStyleSheet(
            "color: #94a3b8;"
        )

        side_layout.addWidget(subtitle)

        side_layout.addSpacing(30)

        buttons = [
            "\ud83c\udfe0 Dashboard",
            "\ud83d\udcdd Essay Analyzer",
            "\ud83d\udd24 Grammar",
            "\ud83d\udcda Content",
            "\ud83c\udfaf Automated Grading",
            "\ud83d\udca1 Feedback"
        ]

        for text in buttons:

            btn = QPushButton(text)

            btn.setStyleSheet("""
                QPushButton {
                    background-color: transparent;
                    text-align: left;
                    padding: 13px;
                }

                QPushButton:hover {
                    background-color: #1e293b;
                }
            """)

            side_layout.addWidget(btn)

        side_layout.addStretch()

        info = QLabel(
            "AI Essay Assessment\nVersion 1.0"
        )

        info.setStyleSheet(
            "color:#64748b;"
        )

        side_layout.addWidget(info)

        # ----------------------------------------------------
        # MAIN CONTENT
        # ----------------------------------------------------

        content = QWidget()

        content_layout = QVBoxLayout(content)

        heading = QLabel(
            "Intelligent Essay Assessment"
        )

        heading.setFont(
            QFont("Arial", 28, QFont.Bold)
        )

        content_layout.addWidget(heading)

        description = QLabel(
            "Analyze grammar, content, coherence and automatically "
            "generate an ML-based grade."
        )

        description.setStyleSheet(
            "color:#94a3b8; font-size:14px;"
        )

        content_layout.addWidget(description)

        content_layout.addSpacing(15)

        # Essay editor
        self.essay_input = QTextEdit()

        self.essay_input.setPlaceholderText(
            "Paste or type your essay here...\n\n"
            "Example:\n"
            "Artificial intelligence is changing education. "
            "It helps students learn faster and provides "
            "personalized learning experiences."
        )

        content_layout.addWidget(
            self.essay_input,
            3
        )

        # Buttons
        button_layout = QHBoxLayout()

        analyze_button = QPushButton(
            "\ud83d\ude80 ANALYZE ESSAY"
        )

        analyze_button.clicked.connect(
            self.analyze_essay
        )

        clear_button = QPushButton(
            "\ud83d\uddd1 CLEAR"
        )

        clear_button.clicked.connect(
            self.clear_all
        )

        button_layout.addWidget(
            analyze_button
        )

        button_layout.addWidget(
            clear_button
        )

        content_layout.addLayout(
            button_layout
        )

        # ----------------------------------------------------
        # SCORE CARDS
        # ----------------------------------------------------

        cards_layout = QHBoxLayout()

        self.grammar_label = self.create_card(
            cards_layout,
            "\ud83d\udd24 Grammar",
            "0%"
        )

        self.content_label = self.create_card(
            cards_layout,
            "\ud83d\udcda Content",
            "0%"
        )

        self.coherence_label = self.create_card(
            cards_layout,
            "\ud83d\udd17 Coherence",
            "0%"
        )

        self.vocab_label = self.create_card(
            cards_layout,
            "\ud83d\udcd6 Vocabulary",
            "0%"
        )

        content_layout.addLayout(
            cards_layout
        )

        # ----------------------------------------------------
        # FINAL SCORE
        # ----------------------------------------------------

        score_frame = QFrame()

        score_frame.setStyleSheet("""
            QFrame {
                background-color: #1e293b;
                border-radius: 12px;
            }
        """)

        score_layout = QVBoxLayout(
            score_frame
        )

        score_title = QLabel(
            "\ud83c\udfaf AUTOMATED ASSESSMENT"
        )

        score_title.setFont(
            QFont("Arial", 16, QFont.Bold)
        )

        score_layout.addWidget(
            score_title
        )

        self.score_label = QLabel(
            "0 / 100"
        )

        self.score_label.setAlignment(
            Qt.AlignCenter
        )

        self.score_label.setFont(
            QFont("Arial", 38, QFont.Bold)
        )

        self.score_label.setStyleSheet(
            "color:#22c55e;"
        )

        score_layout.addWidget(
            self.score_label
        )

        self.grade_label = QLabel(
            "Grade: -"
        )

        self.grade_label.setAlignment(
            Qt.AlignCenter
        )

        self.grade_label.setFont(
            QFont("Arial", 20, QFont.Bold)
        )

        score_layout.addWidget(
            self.grade_label
        )

        content_layout.addWidget(
            score_frame
        )

        # ----------------------------------------------------
        # FEEDBACK
        # ----------------------------------------------------

        self.feedback = QTextEdit()

        self.feedback.setReadOnly(True)

        self.feedback.setPlaceholderText(
            "AI feedback and recommendations will appear here..."
        )

        content_layout.addWidget(
            self.feedback,
            2
        )

        main_layout.addWidget(sidebar)

        main_layout.addWidget(content)

        self.setCentralWidget(
            main_widget
        )


    # ========================================================
    # CREATE SCORE CARD
    # ========================================================

    def create_card(
            self,
            layout,
            title,
            value
    ):

        frame = QFrame()

        frame.setStyleSheet("""
            QFrame {
                background-color: #1e293b;
                border-radius: 10px;
            }
        """)

        card_layout = QVBoxLayout(frame)

        title_label = QLabel(title)

        title_label.setStyleSheet(
            "color:#94a3b8;"
        )

        value_label = QLabel(value)

        value_label.setFont(
            QFont("Arial", 22, QFont.Bold)
        )

        value_label.setStyleSheet(
            "color:#38bdf8;"
        )

        card_layout.addWidget(
            title_label
        )

        card_layout.addWidget(
            value_label
        )

        layout.addWidget(frame)

        return value_label


    # ========================================================
    # ANALYZE ESSAY
    # ========================================================

    def analyze_essay(self):

        text = self.essay_input.toPlainText().strip()

        if len(text) < 30:

            QMessageBox.warning(
                self,
                "Essay Too Short",
                "Please enter an essay with at least 30 characters."
            )

            return

        # Module 1
        grammar, errors = grammar_analysis(text)

        vocabulary = vocabulary_score(text)

        # Module 2
        content = content_score(text)

        coherence = coherence_score(text)

        words = get_words(text)

        sentences = get_sentences(text)

        word_count = len(words)

        sentence_count = max(
            len(sentences),
            1
        )

        avg_sentence_length = (
            word_count / sentence_count
        )

        # Keyword/content feature
        keyword_score = content

        # ML input
        features = np.array([[
            word_count,
            sentence_count,
            avg_sentence_length,
            len(errors),
            vocabulary,
            keyword_score,
            coherence
        ]])

        # Module 3
        ml_score = model.predict(
            features
        )[0]

        # Combine ML + NLP
        final_score = (
            ml_score * 0.40 +
            grammar * 0.20 +
            content * 0.20 +
            coherence * 0.10 +
            vocabulary * 0.10
        )

        final_score = round(
            min(100, max(0, final_score))
        )

        grade = get_grade(
            final_score
        )

        # Update GUI
        self.grammar_label.setText(
            f"{grammar}%"
        )

        self.content_label.setText(
            f"{content}%"
        )

        self.coherence_label.setText(
            f"{coherence}%"
        )

        self.vocab_label.setText(
            f"{vocabulary}%"
        )

        self.score_label.setText(
            f"{final_score} / 100"
        )

        self.grade_label.setText(
            f"Grade: {grade}"
        )

        # Module 4
        feedback = generate_feedback(
            grammar,
            content,
            coherence,
            vocabulary,
            errors
        )

        feedback += (
            "\n\n\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n"
            "\ud83d\udcca ESSAY STATISTICS\n\n"
            f"Words: {word_count}\n"
            f"Sentences: {sentence_count}\n"
            f"Grammar Errors: {len(errors)}\n"
            f"Vocabulary Score: {vocabulary}%\n"
            f"ML Predicted Score: {round(ml_score)}/100"
        )

        self.feedback.setText(
            feedback
        )


    # ========================================================
    # CLEAR
    # ========================================================

    def clear_all(self):

        self.essay_input.clear()

        self.feedback.clear()

        self.grammar_label.setText(
            "0%"
        )

        self.content_label.setText(
            "0%"
        )

        self.coherence_label.setText(
            "0%"
        )

        self.vocab_label.setText(
            "0%"
        )

        self.score_label.setText(
            "0 / 100"
        )

        self.grade_label.setText(
            "Grade: -"
        )


# ============================================================
# RUN APPLICATION
# ============================================================

if __name__ == "__main__":

    app = QApplication(sys.argv)

    window = EssayAssessment()

    window.show()

    sys.exit(
        app.exec()
    )


In [ ]:
import sys
import re
import math
import numpy as np

from PySide6.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
    QTextEdit, QPushButton, QLabel, QFrame, QMessageBox,
    QProgressBar, QStackedWidget
)
from PySide6.QtCore import Qt
from PySide6.QtGui import QFont

from sklearn.ensemble import RandomForestRegressor


# ============================================================
# MACHINE LEARNING MODEL
# ============================================================

def create_ml_model():

    # Training data:
    # [words, sentences, avg_sentence_length,
    #  grammar_errors, vocabulary, keyword_score, coherence]

    X = np.array([
        [80, 5, 16, 8, 40, 45, 45],
        [100, 6, 16, 6, 50, 55, 50],
        [120, 7, 17, 5, 55, 60, 55],
        [150, 8, 19, 4, 60, 70, 65],
        [180, 9, 20, 3, 65, 75, 70],
        [220, 10, 22, 2, 70, 80, 75],
        [250, 11, 23, 2, 75, 85, 80],
        [300, 12, 25, 1, 80, 90, 85],
        [350, 14, 25, 1, 85, 95, 90],
        [400, 16, 25, 0, 90, 98, 95]
    ])

    y = np.array([
        45, 50, 55, 60, 65,
        70, 75, 80, 90, 95
    ])

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    model.fit(X, y)

    return model


model = create_ml_model()


# ============================================================
# NLP FUNCTIONS
# ============================================================

def get_sentences(text):

    sentences = re.split(r'[.!?]+', text)

    return [
        s.strip()
        for s in sentences
        if s.strip()
    ]


def get_words(text):

    return re.findall(r'\b[a-zA-Z]+\b', text.lower())


def grammar_analysis(text):

    words = get_words(text)

    errors = []

    # Simple grammar patterns
    patterns = {
        r'\bi am agree\b': "I agree",
        r'\bhe go\b': "he goes",
        r'\bshe go\b': "she goes",
        r'\bthey goes\b': "they go",
        r'\bhe have\b': "he has",
        r'\bshe have\b': "she has",
        r'\bvery very\b': "very",
        r'\bdid not went\b': "did not go",
        r'\bmore better\b': "better",
        r'\bdiscuss about\b': "discuss"
    }

    for pattern, correction in patterns.items():

        matches = re.findall(pattern, text.lower())

        for match in matches:
            errors.append(
                f"'{match}' → Suggested: '{correction}'"
            )

    # Repeated words
    for i in range(len(words) - 1):

        if words[i] == words[i + 1]:

            errors.append(
                f"Repeated word: '{words[i]}'"
            )

    # Score
    word_count = max(len(words), 1)

    penalty = len(errors) * 5

    score = max(
        0,
        min(100, 100 - penalty)
    )

    return score, errors


def vocabulary_score(text):

    words = get_words(text)

    if not words:
        return 0

    unique_words = set(words)

    richness = len(unique_words) / len(words)

    score = min(100, richness * 140)

    return round(score)


def coherence_score(text):

    sentences = get_sentences(text)

    if len(sentences) < 2:
        return 40

    lengths = [
        len(get_words(sentence))
        for sentence in sentences
    ]

    average = sum(lengths) / len(lengths)

    # Reasonable sentence length
    if 10 <= average <= 25:
        score = 90

    elif 7 <= average <= 30:
        score = 75

    else:
        score = 60

    # Check transition words
    transitions = [
        "however",
        "therefore",
        "moreover",
        "furthermore",
        "because",
        "although",
        "firstly",
        "finally",
        "also",
        "thus",
        "in addition"
    ]

    text_lower = text.lower()

    count = sum(
        1 for word in transitions
        if word in text_lower
    )

    score += min(count * 2, 10)

    return min(score, 100)


def content_score(text):

    words = get_words(text)

    if len(words) < 20:
        return 30

    # Basic content depth estimation
    paragraphs = text.split("\n")

    paragraph_score = min(
        len(paragraphs) * 15,
        40
    )

    word_score = min(
        len(words) / 3,
        40
    )

    keyword_score = 20

    score = (
        paragraph_score +
        word_score +
        keyword_score
    )

    return min(round(score), 100)


def get_grade(score):

    if score >= 90:
        return "A+"

    elif score >= 80:
        return "A"

    elif score >= 70:
        return "B"

    elif score >= 60:
        return "C"

    elif score >= 50:
        return "D"

    else:
        return "F"


def generate_feedback(
        grammar,
        content,
        coherence,
        vocabulary,
        errors
):

    feedback = []

    # Strengths
    feedback.append("✓ STRENGTHS")

    if grammar >= 80:
        feedback.append(
            "• Good grammatical accuracy."
        )
    else:
        feedback.append(
            "• Grammar needs improvement."
        )

    if vocabulary >= 75:
        feedback.append(
            "• Good vocabulary usage."
        )
    else:
        feedback.append(
            "• Try using a wider range of vocabulary."
        )

    if content >= 75:
        feedback.append(
            "• Essay contains relevant content."
        )
    else:
        feedback.append(
            "• Add more supporting ideas and examples."
        )

    if coherence >= 75:
        feedback.append(
            "• Ideas are reasonably well connected."
        )
    else:
        feedback.append(
            "• Improve connections between paragraphs."
        )

    feedback.append("")
    feedback.append("⚠ AREAS TO IMPROVE")

    if errors:

        for error in errors[:5]:

            feedback.append(
                "• " + error
            )

    else:

        feedback.append(
            "• No major grammar problems detected."
        )

    if coherence < 75:

        feedback.append(
            "• Use transition words such as however, therefore and moreover."
        )

    if vocabulary < 75:

        feedback.append(
            "• Avoid repeating the same words."
        )

    if content < 75:

        feedback.append(
            "• Add examples, arguments and explanations."
        )

    feedback.append("")
    feedback.append("💡 RECOMMENDATIONS")

    feedback.append(
        "1. Review grammatical mistakes."
    )

    feedback.append(
        "2. Add stronger supporting examples."
    )

    feedback.append(
        "3. Improve paragraph transitions."
    )

    feedback.append(
        "4. Use more academic vocabulary."
    )

    feedback.append(
        "5. Write a clear conclusion."
    )

    return "\n".join(feedback)


# ============================================================
# MAIN GUI
# ============================================================

class EssayAssessment(QMainWindow):

    def __init__(self):

        super().__init__()

        self.setWindowTitle(
            "Intelligent Essay Assessment - NLP & ML"
        )

        self.setMinimumSize(1200, 750)

        self.setStyleSheet("""
            QMainWindow {
                background-color: #0f172a;
            }

            QLabel {
                color: white;
            }

            QTextEdit {
                background-color: #1e293b;
                color: white;
                border: 1px solid #334155;
                border-radius: 10px;
                padding: 15px;
                font-size: 15px;
            }

            QPushButton {
                background-color: #2563eb;
                color: white;
                border: none;
                border-radius: 8px;
                padding: 12px 20px;
                font-size: 14px;
                font-weight: bold;
            }

            QPushButton:hover {
                background-color: #3b82f6;
            }

            QProgressBar {
                border: none;
                border-radius: 7px;
                background-color: #334155;
                height: 12px;
            }

            QProgressBar::chunk {
                background-color: #22c55e;
                border-radius: 7px;
            }
        """)

        self.setup_ui()


    # ========================================================
    # UI
    # ========================================================

    def setup_ui(self):

        main_widget = QWidget()

        main_layout = QHBoxLayout(
            main_widget
        )

        # ----------------------------------------------------
        # SIDEBAR
        # ----------------------------------------------------

        sidebar = QFrame()

        sidebar.setFixedWidth(220)

        sidebar.setStyleSheet("""
            QFrame {
                background-color: #111827;
                border-radius: 10px;
            }
        """)

        side_layout = QVBoxLayout(sidebar)

        title = QLabel("🧠 EssayAI")

        title.setFont(
            QFont("Arial", 22, QFont.Bold)
        )

        side_layout.addWidget(title)

        subtitle = QLabel(
            "NLP + Machine Learning"
        )

        subtitle.setStyleSheet(
            "color: #94a3b8;"
        )

        side_layout.addWidget(subtitle)

        side_layout.addSpacing(30)

        buttons = [
            "🏠 Dashboard",
            "📝 Essay Analyzer",
            "🔤 Grammar",
            "📚 Content",
            "🎯 Automated Grading",
            "💡 Feedback"
        ]

        for text in buttons:

            btn = QPushButton(text)

            btn.setStyleSheet("""
                QPushButton {
                    background-color: transparent;
                    text-align: left;
                    padding: 13px;
                }

                QPushButton:hover {
                    background-color: #1e293b;
                }
            """)

            side_layout.addWidget(btn)

        side_layout.addStretch()

        info = QLabel(
            "AI Essay Assessment\nVersion 1.0"
        )

        info.setStyleSheet(
            "color:#64748b;"
        )

        side_layout.addWidget(info)

        # ----------------------------------------------------
        # MAIN CONTENT
        # ----------------------------------------------------

        content = QWidget()

        content_layout = QVBoxLayout(content)

        heading = QLabel(
            "Intelligent Essay Assessment"
        )

        heading.setFont(
            QFont("Arial", 28, QFont.Bold)
        )

        content_layout.addWidget(heading)

        description = QLabel(
            "Analyze grammar, content, coherence and automatically "
            "generate an ML-based grade."
        )

        description.setStyleSheet(
            "color:#94a3b8; font-size:14px;"
        )

        content_layout.addWidget(description)

        content_layout.addSpacing(15)

        # Essay editor
        self.essay_input = QTextEdit()

        self.essay_input.setPlaceholderText(
            "Paste or type your essay here...\n\n"
            "Example:\n"
            "Artificial intelligence is changing education. "
            "It helps students learn faster and provides "
            "personalized learning experiences."
        )

        content_layout.addWidget(
            self.essay_input,
            3
        )

        # Buttons
        button_layout = QHBoxLayout()

        analyze_button = QPushButton(
            "🚀 ANALYZE ESSAY"
        )

        analyze_button.clicked.connect(
            self.analyze_essay
        )

        clear_button = QPushButton(
            "🗑 CLEAR"
        )

        clear_button.clicked.connect(
            self.clear_all
        )

        button_layout.addWidget(
            analyze_button
        )

        button_layout.addWidget(
            clear_button
        )

        content_layout.addLayout(
            button_layout
        )

        # ----------------------------------------------------
        # SCORE CARDS
        # ----------------------------------------------------

        cards_layout = QHBoxLayout()

        self.grammar_label = self.create_card(
            cards_layout,
            "🔤 Grammar",
            "0%"
        )

        self.content_label = self.create_card(
            cards_layout,
            "📚 Content",
            "0%"
        )

        self.coherence_label = self.create_card(
            cards_layout,
            "🔗 Coherence",
            "0%"
        )

        self.vocab_label = self.create_card(
            cards_layout,
            "📖 Vocabulary",
            "0%"
        )

        content_layout.addLayout(
            cards_layout
        )

        # ----------------------------------------------------
        # FINAL SCORE
        # ----------------------------------------------------

        score_frame = QFrame()

        score_frame.setStyleSheet("""
            QFrame {
                background-color: #1e293b;
                border-radius: 12px;
            }
        """)

        score_layout = QVBoxLayout(
            score_frame
        )

        score_title = QLabel(
            "🎯 AUTOMATED ASSESSMENT"
        )

        score_title.setFont(
            QFont("Arial", 16, QFont.Bold)
        )

        score_layout.addWidget(
            score_title
        )

        self.score_label = QLabel(
            "0 / 100"
        )

        self.score_label.setAlignment(
            Qt.AlignCenter
        )

        self.score_label.setFont(
            QFont("Arial", 38, QFont.Bold)
        )

        self.score_label.setStyleSheet(
            "color:#22c55e;"
        )

        score_layout.addWidget(
            self.score_label
        )

        self.grade_label = QLabel(
            "Grade: -"
        )

        self.grade_label.setAlignment(
            Qt.AlignCenter
        )

        self.grade_label.setFont(
            QFont("Arial", 20, QFont.Bold)
        )

        score_layout.addWidget(
            self.grade_label
        )

        content_layout.addWidget(
            score_frame
        )

        # ----------------------------------------------------
        # FEEDBACK
        # ----------------------------------------------------

        self.feedback = QTextEdit()

        self.feedback.setReadOnly(True)

        self.feedback.setPlaceholderText(
            "AI feedback and recommendations will appear here..."
        )

        content_layout.addWidget(
            self.feedback,
            2
        )

        main_layout.addWidget(sidebar)

        main_layout.addWidget(content)

        self.setCentralWidget(
            main_widget
        )


    # ========================================================
    # CREATE SCORE CARD
    # ========================================================

    def create_card(
            self,
            layout,
            title,
            value
    ):

        frame = QFrame()

        frame.setStyleSheet("""
            QFrame {
                background-color: #1e293b;
                border-radius: 10px;
            }
        """)

        card_layout = QVBoxLayout(frame)

        title_label = QLabel(title)

        title_label.setStyleSheet(
            "color:#94a3b8;"
        )

        value_label = QLabel(value)

        value_label.setFont(
            QFont("Arial", 22, QFont.Bold)
        )

        value_label.setStyleSheet(
            "color:#38bdf8;"
        )

        card_layout.addWidget(
            title_label
        )

        card_layout.addWidget(
            value_label
        )

        layout.addWidget(frame)

        return value_label


    # ========================================================
    # ANALYZE ESSAY
    # ========================================================

    def analyze_essay(self):

        text = self.essay_input.toPlainText().strip()

        if len(text) < 30:

            QMessageBox.warning(
                self,
                "Essay Too Short",
                "Please enter an essay with at least 30 characters."
            )

            return

        # Module 1
        grammar, errors = grammar_analysis(text)

        vocabulary = vocabulary_score(text)

        # Module 2
        content = content_score(text)

        coherence = coherence_score(text)

        words = get_words(text)

        sentences = get_sentences(text)

        word_count = len(words)

        sentence_count = max(
            len(sentences),
            1
        )

        avg_sentence_length = (
            word_count / sentence_count
        )

        # Keyword/content feature
        keyword_score = content

        # ML input
        features = np.array([[
            word_count,
            sentence_count,
            avg_sentence_length,
            len(errors),
            vocabulary,
            keyword_score,
            coherence
        ]])

        # Module 3
        ml_score = model.predict(
            features
        )[0]

        # Combine ML + NLP
        final_score = (
            ml_score * 0.40 +
            grammar * 0.20 +
            content * 0.20 +
            coherence * 0.10 +
            vocabulary * 0.10
        )

        final_score = round(
            min(100, max(0, final_score))
        )

        grade = get_grade(
            final_score
        )

        # Update GUI
        self.grammar_label.setText(
            f"{grammar}%"
        )

        self.content_label.setText(
            f"{content}%"
        )

        self.coherence_label.setText(
            f"{coherence}%"
        )

        self.vocab_label.setText(
            f"{vocabulary}%"
        )

        self.score_label.setText(
            f"{final_score} / 100"
        )

        self.grade_label.setText(
            f"Grade: {grade}"
        )

        # Module 4
        feedback = generate_feedback(
            grammar,
            content,
            coherence,
            vocabulary,
            errors
        )

        feedback += (
            "\n\n────────────────────────────\n"
            "📊 ESSAY STATISTICS\n\n"
            f"Words: {word_count}\n"
            f"Sentences: {sentence_count}\n"
            f"Grammar Errors: {len(errors)}\n"
            f"Vocabulary Score: {vocabulary}%\n"
            f"ML Predicted Score: {round(ml_score)}/100"
        )

        self.feedback.setText(
            feedback
        )


    # ========================================================
    # CLEAR
    # ========================================================

    def clear_all(self):

        self.essay_input.clear()

        self.feedback.clear()

        self.grammar_label.setText(
            "0%"
        )

        self.content_label.setText(
            "0%"
        )

        self.coherence_label.setText(
            "0%"
        )

        self.vocab_label.setText(
            "0%"
        )

        self.score_label.setText(
            "0 / 100"
        )

        self.grade_label.setText(
            "Grade: -"
        )


# ============================================================
# RUN APPLICATION
# ============================================================

if __name__ == "__main__":

    app = QApplication(sys.argv)

    window = EssayAssessment()

    window.show()

    sys.exit(
        app.exec()
    )

In [1]:
!pip install -q gradio nltk scikit-learn pandas numpy matplotlib textblob

In [2]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [4]:
import re
import numpy as np
import pandas as pd
import gradio as gr

from sklearn.ensemble import RandomForestRegressor
from textblob import TextBlob


# ============================================================
# 1. MACHINE LEARNING MODEL
# ============================================================

# Demonstration training data
# Features:
# words, sentences, grammar_errors,
# vocabulary, content, coherence

X = np.array([
    [50, 4, 10, 40, 40, 40],
    [80, 5, 8, 50, 50, 50],
    [100, 6, 7, 55, 60, 55],
    [130, 7, 5, 60, 65, 65],
    [160, 8, 4, 65, 70, 70],
    [200, 10, 3, 70, 75, 75],
    [250, 12, 2, 75, 80, 80],
    [300, 14, 2, 80, 85, 85],
    [350, 16, 1, 90, 90, 90],
    [400, 18, 0, 95, 95, 95]
])

y = np.array([
    40, 45, 50, 55, 60,
    68, 75, 80, 90, 96
])

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)


# ============================================================
# 2. TEXT PROCESSING
# ============================================================

def get_words(text):

    return re.findall(
        r'\b[a-zA-Z]+\b',
        text.lower()
    )


def get_sentences(text):

    return [
        s.strip()
        for s in re.split(
            r'[.!?]+',
            text
        )
        if s.strip()
    ]


# ============================================================
# MODULE 1
# GRAMMAR & LANGUAGE ANALYSIS
# ============================================================

def grammar_analysis(text):

    errors = []

    # Basic grammar patterns
    corrections = {

        r'\bhe go\b':
            "He goes",

        r'\bshe go\b':
            "She goes",

        r'\bthey goes\b':
            "They go",

        r'\bhe have\b':
            "He has",

        r'\bshe have\b':
            "She has",

        r'\bi am agree\b':
            "I agree",

        r'\bmore better\b':
            "better",

        r'\bvery very\b':
            "very",

        r'\bdid not went\b':
            "did not go",

        r'\bdiscuss about\b':
            "discuss"
    }

    for pattern, correction in corrections.items():

        matches = re.findall(
            pattern,
            text.lower()
        )

        for match in matches:

            errors.append(
                f"❌ {match} → {correction}"
            )

    words = get_words(text)

    # repeated words
    for i in range(len(words) - 1):

        if words[i] == words[i + 1]:

            errors.append(
                f"⚠ Repeated word: {words[i]}"
            )

    # Grammar score
    penalty = len(errors) * 5

    grammar_score = max(
        0,
        100 - penalty
    )

    return grammar_score, errors


# ============================================================
# VOCABULARY ANALYSIS
# ============================================================

def vocabulary_analysis(text):

    words = get_words(text)

    if len(words) == 0:

        return 0

    unique_words = set(words)

    richness = (
        len(unique_words) /
        len(words)
    )

    score = min(
        100,
        round(richness * 150)
    )

    return score


# ============================================================
# MODULE 2
# CONTENT EVALUATION
# ============================================================

def content_analysis(text):

    words = get_words(text)

    paragraphs = [
        p for p in text.split("\n")
        if p.strip()
    ]

    word_score = min(
        40,
        len(words) / 5
    )

    paragraph_score = min(
        30,
        len(paragraphs) * 10
    )

    topic_score = 30

    score = (
        word_score +
        paragraph_score +
        topic_score
    )

    return round(
        min(100, score)
    )


# ============================================================
# COHERENCE ANALYSIS
# ============================================================

def coherence_analysis(text):

    sentences = get_sentences(text)

    if len(sentences) <= 1:

        return 40

    transition_words = [

        "however",
        "therefore",
        "moreover",
        "furthermore",
        "because",
        "although",
        "firstly",
        "secondly",
        "finally",
        "also",
        "thus",
        "in addition"
    ]

    transition_count = 0

    lower_text = text.lower()

    for word in transition_words:

        if word in lower_text:

            transition_count += 1

    sentence_lengths = [

        len(get_words(s))
        for s in sentences
    ]

    average_length = np.mean(
        sentence_lengths
    )

    # Ideal sentence length
    if 10 <= average_length <= 25:

        score = 80

    elif 7 <= average_length <= 30:

        score = 70

    else:

        score = 55

    score += min(
        transition_count * 3,
        20
    )

    return min(
        100,
        round(score)
    )


# ============================================================
# MODULE 3
# AUTOMATED GRADING
# ============================================================

def automated_grading(
        words,
        sentences,
        grammar,
        vocabulary,
        content,
        coherence
):

    sentence_count = max(
        sentences,
        1
    )

    features = np.array([[
        words,
        sentence_count,
        100 - grammar,
        vocabulary,
        content,
        coherence
    ]])

    ml_score = model.predict(
        features
    )[0]

    # Combine ML prediction
    # with NLP scores

    final_score = (

        ml_score * 0.40 +

        grammar * 0.20 +

        content * 0.20 +

        coherence * 0.10 +

        vocabulary * 0.10
    )

    return round(
        min(100, max(0, final_score))
    )


# ============================================================
# GRADE
# ============================================================

def get_grade(score):

    if score >= 90:

        return "A+"

    elif score >= 80:

        return "A"

    elif score >= 70:

        return "B"

    elif score >= 60:

        return "C"

    elif score >= 50:

        return "D"

    else:

        return "F"


# ============================================================
# MODULE 4
# FEEDBACK & RECOMMENDATIONS
# ============================================================

def generate_feedback(
        grammar,
        vocabulary,
        content,
        coherence,
        errors
):

    feedback = []

    feedback.append(
        "## 🌟 STRENGTHS"
    )

    if grammar >= 80:

        feedback.append(
            "✅ Good grammatical accuracy."
        )

    if vocabulary >= 80:

        feedback.append(
            "✅ Strong vocabulary usage."
        )

    if content >= 80:

        feedback.append(
            "✅ Good content development."
        )

    if coherence >= 80:

        feedback.append(
            "✅ Good logical flow."
        )

    feedback.append(
        "\n## ⚠️ AREAS TO IMPROVE"
    )

    if errors:

        for error in errors[:5]:

            feedback.append(error)

    else:

        feedback.append(
            "✅ No major grammar issues detected."
        )

    if vocabulary < 70:

        feedback.append(
            "⚠️ Use a wider range of vocabulary."
        )

    if content < 70:

        feedback.append(
            "⚠️ Add more examples and supporting ideas."
        )

    if coherence < 70:

        feedback.append(
            "⚠️ Improve connections between paragraphs."
        )

    feedback.append(
        "\n## 💡 RECOMMENDATIONS"
    )

    feedback.append(
        "1. Improve sentence structure."
    )

    feedback.append(
        "2. Add supporting examples."
    )

    feedback.append(
        "3. Use transition words."
    )

    feedback.append(
        "4. Avoid repeated words."
    )

    feedback.append(
        "5. Strengthen the conclusion."
    )

    return "\n".join(feedback)


# ============================================================
# MAIN ANALYSIS FUNCTION
# ============================================================

def analyze_essay(
        essay
):

    if essay is None or len(essay.strip()) < 30:

        return (
            "⚠️ Please enter an essay with at least 30 characters.",
            "",
            "",
            "",
            "",
            ""
        )

    # Words and sentences

    words = get_words(essay)

    sentences = get_sentences(essay)

    word_count = len(words)

    sentence_count = len(sentences)

    # Module 1

    grammar, errors = grammar_analysis(
        essay
    )

    vocabulary = vocabulary_analysis(
        essay
    )

    # Module 2

    content = content_analysis(
        essay
    )

    coherence = coherence_analysis(
        essay
    )

    # Module 3

    final_score = automated_grading(

        word_count,
        sentence_count,
        grammar,
        vocabulary,
        content,
        coherence
    )

    grade = get_grade(
        final_score
    )

    # Module 4

    feedback = generate_feedback(

        grammar,
        vocabulary,
        content,
        coherence,
        errors
    )

    # Scorecard

    scorecard = f"""
# 🎯 AI ESSAY SCORECARD

## Overall Score

# **{final_score} / 100**

### Grade: **{grade}**

---

| Evaluation | Score |
|---|---:|
| 🔤 Grammar | {grammar}% |
| 📚 Content | {content}% |
| 🔗 Coherence | {coherence}% |
| 📖 Vocabulary | {vocabulary}% |
| 🤖 ML Assessment | {final_score}% |

---

### 📊 Essay Statistics

**Words:** {word_count}

**Sentences:** {sentence_count}

**Grammar Issues:** {len(errors)}

"""

    # Grammar result

    grammar_result = f"""
# 🔤 GRAMMAR & LANGUAGE ANALYSIS

### Grammar Score: **{grammar}%**

### Vocabulary Score: **{vocabulary}%**

### Detected Issues

"""

    if errors:

        grammar_result += "\n".join(
            errors
        )

    else:

        grammar_result += (
            "✅ No major grammar errors detected."
        )

    # Content

    content_result = f"""
# 📚 CONTENT & COHERENCE

### Content Score: **{content}%**

### Coherence Score: **{coherence}%**

### Analysis

The essay contains **{word_count} words**
and **{sentence_count} sentences**.

"""

    if content >= 80:

        content_result += (
            "✅ The essay has good content development."
        )

    elif content >= 60:

        content_result += (
            "⚠️ Content is acceptable but can be expanded."
        )

    else:

        content_result += (
            "❌ More relevant ideas and examples are required."
        )

    # Grading

    grading_result = f"""
# 🎯 AUTOMATED GRADING

## Predicted Score

# **{final_score}/100**

## Grade

# **{grade}**

### Machine Learning

The Random Forest model uses NLP-derived
features to estimate the essay score.

Features used:

- Word count
- Sentence count
- Grammar quality
- Vocabulary richness
- Content score
- Coherence score
"""

    return (
        scorecard,
        grammar_result,
        content_result,
        grading_result,
        feedback,
        f"{final_score}/100"
    )


# ============================================================
# GRADIO GUI
# ============================================================

custom_css = """

body {
    background-color: #0f172a;
}

.gradio-container {
    max-width: 1200px !important;
}

.title {
    text-align: center;
}

"""

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="blue"
    ),
    css=custom_css
) as demo:

    gr.Markdown(
        """
# 🧠 Intelligent Essay Assessment
### NLP + Machine Learning Based Essay Evaluation

**Analyze → Evaluate → Grade → Improve**
        """
    )

    with gr.Row():

        with gr.Column(
            scale=2
        ):

            essay = gr.Textbox(
                label="📝 Enter Your Essay",
                placeholder=(
                    "Paste or type your essay here..."
                ),
                lines=18
            )

            analyze_button = gr.Button(
                "🚀 ANALYZE ESSAY",
                variant="primary"
            )

            clear_button = gr.ClearButton(
                components=[essay],
                value="🗑 CLEAR"
            )

        with gr.Column(
            scale=1
        ):

            final_score = gr.Markdown(
                """
# 🎯 SCORE

Enter an essay and click
**Analyze Essay**.
                """
            )

    gr.Markdown(
        "## 📊 AI Assessment Dashboard"
    )

    with gr.Tabs():

        # MODULE 1

        with gr.Tab(
            "🔤 Module 1 - Grammar"
        ):

            grammar_output = gr.Markdown()

        # MODULE 2

        with gr.Tab(
            "📚 Module 2 - Content"
        ):

            content_output = gr.Markdown()

        # MODULE 3

        with gr.Tab(
            "🎯 Module 3 - Grading"
        ):

            grading_output = gr.Markdown()

        # MODULE 4

        with gr.Tab(
            "💡 Module 4 - Feedback"
        ):

            feedback_output = gr.Markdown()

        # COMPLETE SCORECARD

        with gr.Tab(
            "📊 Complete Scorecard"
        ):

            scorecard_output = gr.Markdown()

    analyze_button.click(

        fn=analyze_essay,

        inputs=essay,

        outputs=[
            scorecard_output,
            grammar_output,
            content_output,
            grading_output,
            feedback_output,
            final_score
        ]
    )


# ============================================================
# LAUNCH
# ============================================================

demo.launch(
    share=True
)

/tmp/ipykernel_1842/1062984866.py:697: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c69650be7f736831aa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
